# EE200 Summer 2025 - Signal Processing Project
# Image Transforms, Audio Analysis, and Frequency Domain Processing

This notebook covers:
1. Basic Image Operations (resize, crop, rotate)
2. 2D Discrete Fourier Transform (DFT)
3. Frequency Domain Filtering (LPF/HPF)
4. Audio Loading and Waveform Visualization
5. Time-Frequency Analysis (Spectrogram)

In [ ]:
# Cell 1: Setup and Imports
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import librosa
import librosa.display
from scipy.fft import fft2, ifft2, fftshift
import os

# Set up inline plotting
%matplotlib inline
plt.style.use('default')

# Get current directory for file paths
base_path = os.path.dirname(os.path.abspath('__file__')) if '__file__' in locals() else os.getcwd()
print(f"Working directory: {base_path}")

---
## Part A: Image Processing

In [ ]:
# Cell 2: Load and Display Original Images
# Load grayscale images
cat_img = Image.open('cat_gray.jpg')
dog_img = Image.open('dog_gray.jpg')

# Convert to numpy arrays for processing
cat_array = np.array(cat_img)
dog_array = np.array(dog_img)

print(f"Cat image shape: {cat_array.shape}")
print(f"Dog image shape: {dog_array.shape}")
print(f"Cat dtype: {cat_array.dtype}")
print(f"Dog dtype: {dog_array.dtype}")

# Display original images
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cat_img, cmap='gray')
axes[0].set_title('Cat (Grayscale)')
axes[0].axis('off')

axes[1].imshow(dog_img, cmap='gray')
axes[1].set_title('Dog (Grayscale)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 3: Basic Image Operations - Resize, Crop, Rotate
# Resize both images to 200x200
cat_resized = cat_img.resize((200, 200))
dog_resized = dog_img.resize((200, 200))

# Crop: (left, upper, right, lower) coordinates
cat_cropped = cat_img.crop((50, 50, 200, 200))
dog_cropped = dog_img.crop((50, 50, 200, 200))

# Rotate by 45 degrees counter-clockwise
cat_rotated = cat_img.rotate(45)
dog_rotated = dog_img.rotate(45)

# Display all operations for cat image
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(cat_img, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(cat_resized, cmap='gray')
axes[1].set_title('Resized (200x200)')
axes[1].axis('off')

axes[2].imshow(cat_cropped, cmap='gray')
axes[2].set_title('Cropped (50-200)')
axes[2].axis('off')

axes[3].imshow(cat_rotated, cmap='gray')
axes[3].set_title('Rotated (45°)')
axes[3].axis('off')

plt.tight_layout()
plt.show()

print("\nCat image operations completed!")

In [ ]:
# Cell 4: 2D Discrete Fourier Transform (DFT)
# Compute 2D FFT for cat image
cat_fft = fft2(cat_array)
dog_fft = fft2(dog_array)

# Shift zero frequency to center
cat_fft_shift = fftshift(cat_fft)
dog_fft_shift = fftshift(dog_fft)

# Compute magnitude spectrum (log scale for better visualization)
cat_magnitude = np.log(1 + np.abs(cat_fft_shift))
dog_magnitude = np.log(1 + np.abs(dog_fft_shift))

# Compute phase spectrum
cat_phase = np.angle(cat_fft_shift)
dog_phase = np.angle(dog_fft_shift)

print(f"FFT shape: {cat_fft.shape}")
print(f"Max magnitude (log): {cat_magnitude.max():.2f}")
print(f"Phase range: [{cat_phase.min():.2f}, {cat_phase.max():.2f}] radians")

# Display magnitude spectra
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(cat_magnitude, cmap='gray')
axes[0].set_title('Cat - Magnitude Spectrum (Log)')
axes[0].axis('off')

axes[1].imshow(dog_magnitude, cmap='gray')
axes[1].set_title('Dog - Magnitude Spectrum (Log)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Display Phase Spectra
# Display phase spectra
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im1 = axes[0].imshow(cat_phase, cmap='twilight', aspect='auto')
axes[0].set_title('Cat - Phase Spectrum')
axes[0].axis('off')
plt.colorbar(im1, ax=axes[0], fraction=0.046)

im2 = axes[1].imshow(dog_phase, cmap='twilight', aspect='auto')
axes[1].set_title('Dog - Phase Spectrum')
axes[1].axis('off')
plt.colorbar(im2, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

print("Phase spectra show the phase angle of each frequency component.")
print("The central area contains the DC component and low frequencies.")

In [ ]:
# Cell 6: Frequency Domain Filtering - Create Filter Masks with Multiple Cutoffs
def create_ideal_filter(shape, cutoff, filter_type='low'):
    """
    Create an ideal filter in frequency domain.

    Parameters:
    - shape: tuple (M, N) image dimensions
    - cutoff: cutoff frequency D0
    - filter_type: 'low' for LPF, 'high' for HPF

    Returns:
    - filter mask of same shape
    """
    M, N = shape
    # Create coordinate grids centered at zero
    u = np.arange(-M//2, M//2)
    v = np.arange(-N//2, N//2)
    V, U = np.meshgrid(v, u)

    # Compute distance from center (frequency radius)
    D = np.sqrt(U**2 + V**2)

    if filter_type == 'low':
        # Ideal Low-Pass Filter: 1 inside radius, 0 outside
        H = (D <= cutoff).astype(float)
    else:  # high-pass
        # Ideal High-Pass Filter: 0 inside radius, 1 outside
        H = (D > cutoff).astype(float)

    return H

# Get image dimensions
M, N = cat_array.shape

# Create filters at THREE different cutoffs for visible effects
cutoff_conservative = min(M, N) // 4   # 25% - subtle blur
cutoff_default = min(M, N) // 8         # 12.5% - moderate effect
cutoff_aggressive = min(M, N) // 16     # 6.25% - strong blur

print(f"Image dimensions: {M} x {N}")
print(f"Conservative cutoff (25%): {cutoff_conservative} pixels - subtle blur")
print(f"Default cutoff (12.5%): {cutoff_default} pixels - moderate effect")
print(f"Aggressive cutoff (6.25%): {cutoff_aggressive} pixels - strong blur")

# Create LPF masks at all cutoffs
lpf_conservative = create_ideal_filter((M, N), cutoff_conservative, 'low')
lpf_default = create_ideal_filter((M, N), cutoff_default, 'low')
lpf_aggressive = create_ideal_filter((M, N), cutoff_aggressive, 'low')

# Create HPF mask (edges look similar at different cutoffs)
hpf_default = create_ideal_filter((M, N), cutoff_default, 'high')

# Visualize all filter masks for comparison
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(lpf_conservative, cmap='gray')
axes[0].set_title(f'LPF Conservative\n(D0={cutoff_conservative}, 25%)')
axes[0].axis('off')

axes[1].imshow(lpf_default, cmap='gray')
axes[1].set_title(f'LPF Default\n(D0={cutoff_default}, 12.5%)')
axes[1].axis('off')

axes[2].imshow(lpf_aggressive, cmap='gray')
axes[2].set_title(f'LPF Aggressive\n(D0={cutoff_aggressive}, 6.25%)')
axes[2].axis('off')

axes[3].imshow(hpf_default, cmap='gray')
axes[3].set_title(f'HPF Default\n(D0={cutoff_default})')
axes[3].axis('off')

plt.tight_layout()
plt.show()

print("\nFilter masks show which frequencies are passed (white = 1) or blocked (black = 0)")

In [ ]:
# Cell 7: Apply Filters with Multiple Cutoff Comparison
# Apply LPF at different cutoffs to cat image
cat_lpf_conservative = np.real(ifft2(fftshift(cat_fft_shift * lpf_conservative)))
cat_lpf_default = np.real(ifft2(fftshift(cat_fft_shift * lpf_default)))
cat_lpf_aggressive = np.real(ifft2(fftshift(cat_fft_shift * lpf_aggressive)))

# Apply HPF to cat image
cat_hpf = np.real(ifft2(fftshift(cat_fft_shift * hpf_default)))

# Apply LPF at default cutoff to dog image
dog_lpf_default = np.real(ifft2(fftshift(dog_fft_shift * lpf_default)))

# Apply HPF to dog image
dog_hpf = np.real(ifft2(fftshift(dog_fft_shift * hpf_default)))

print("Filtering completed successfully!")
print(f"\nCat LPF results:")
print(f"  Conservative (25%): range [{cat_lpf_conservative.min():.2f}, {cat_lpf_conservative.max():.2f}]")
print(f"  Default (12.5%): range [{cat_lpf_default.min():.2f}, {cat_lpf_default.max():.2f}]")
print(f"  Aggressive (6.25%): range [{cat_lpf_aggressive.min():.2f}, {cat_lpf_aggressive.max():.2f}]")
print(f"\nCat HPF result:")
print(f"  Default (12.5%): range [{cat_hpf.min():.2f}, {cat_hpf.max():.2f}]")

In [ ]:
# Cell 8b: Rotate Image and Compare 2D DFT Spectra
# Rotate the cat image anti-clockwise 90 degrees
cat_rotated_90 = cat_img.rotate(90)

# Convert to numpy array
cat_rotated_array = np.array(cat_rotated_90)

# Compute 2D DFT of rotated image
cat_rotated_fft = fft2(cat_rotated_array)
cat_rotated_fft_shift = fftshift(cat_rotated_fft)

# Compute magnitude spectrum (log scale)
cat_rotated_magnitude = np.log(1 + np.abs(cat_rotated_fft_shift))

print("=" * 70)
print("ROTATED IMAGE ANALYSIS - 90° Anti-Clockwise Rotation")
print("=" * 70)
print(f"Original shape: {cat_array.shape}")
print(f"Rotated shape: {cat_rotated_array.shape}")

# Display comparison: Original, Rotated, Original FFT, Rotated FFT
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Original image
axes[0, 0].imshow(cat_array, cmap='gray')
axes[0, 0].set_title('Original Cat Image')
axes[0, 0].axis('off')

# Rotated image
axes[0, 1].imshow(cat_rotated_array, cmap='gray')
axes[0, 1].set_title('Rotated (90° Anti-Clockwise)')
axes[0, 1].axis('off')

# Original magnitude spectrum
axes[1, 0].imshow(cat_magnitude, cmap='gray')
axes[1, 0].set_title('Original Magnitude Spectrum')
axes[1, 0].axis('off')

# Rotated magnitude spectrum
axes[1, 1].imshow(cat_rotated_magnitude, cmap='gray')
axes[1, 1].set_title('Rotated Magnitude Spectrum')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("OBSERVATIONS - Comparing Original vs Rotated FFT Spectra")
print("=" * 70)
print("""
1. ROTATION EFFECT: When an image is rotated by θ, its Fourier transform
   is also rotated by the same angle θ in the frequency domain.

2. SPECTRAL PATTERN: The rotated FFT shows the same energy distribution
   as the original, but rotated by 90° to match the spatial rotation.

3. MAGNITUDE RELATIONSHIP: |F(u,v)| = |F_rotated(u,v)| after rotation
   - The magnitude of the Fourier transform is rotation-invariant
   - Only the orientation changes, not the magnitude distribution

4. ENERGY CONSERVATION: Both spectra have the same total energy
   - Sum of squared magnitudes is preserved under rotation
   - This demonstrates Parseval's theorem in 2D

5. PRACTICAL IMPLICATION: FFT-based processing (filtering, etc.)
   works the same way regardless of image orientation.
""")

In [ ]:
# Cell 10: Load Audio - Prepare for Audio Processing
# Load the audio file first (before any audio processing)
audio_path = 'song_with_2piccolo.wav'
y, sr = librosa.load(audio_path, sr=None)  # sr=None keeps original sampling rate

# Display audio metadata
duration = len(y) / sr
print("=" * 70)
print("AUDIO LOADING - Song with 2 Piccolo")
print("=" * 70)
print(f"Audio file: {audio_path}")
print(f"Sampling rate: {sr} Hz")
print(f"Duration: {duration:.2f} seconds")
print(f"Number of samples: {len(y)}")
print(f"Signal dtype: {y.dtype}")
print(f"Signal range: [{y.min():.4f}, {y.max():.4f}]")

In [ ]:
# Cell 11: Audio Restoration - Frequency Domain Filtering
# Apply FFT-based LPF and HPF to audio signal

print("=" * 70)
print("AUDIO RESTORATION - Frequency Domain Filtering")
print("=" * 70)

# Parameters for audio filtering
audio_D0 = 1000  # Cutoff frequency in Hz

print(f"""
AUDIO FILTERING PARAMETERS:
==========================
Original sampling rate: {sr} Hz
Nyquist frequency: {sr/2} Hz
LPF Cutoff frequency: {audio_D0} Hz
""")

# Compute FFT of audio signal
y_fft = np.fft.rfft(y)
frequencies_audio = np.fft.rfftfreq(len(y), 1/sr)

# Create audio filter masks
audio_lpf_mask = (np.abs(frequencies_audio) <= audio_D0).astype(float)
audio_hpf_mask = (np.abs(frequencies_audio) > audio_D0).astype(float)

# Apply filters in frequency domain
y_lpf = np.fft.irfft(y_fft * audio_lpf_mask)
y_hpf = np.fft.irfft(y_fft * audio_hpf_mask)

print(f"✓ Applied LPF with cutoff {audio_D0} Hz")
print(f"✓ Applied HPF with cutoff {audio_D0} Hz")

# Compare original and filtered audio
print("\n--- Audio Restoration Comparison ---")
print(f"Original audio range: [{y.min():.4f}, {y.max():.4f}]")
print(f"LPF filtered range: [{y_lpf.min():.4f}, {y_lpf.max():.4f}]")
print(f"HPF filtered range: [{y_hpf.min():.4f}, {y_hpf.max():.4f}]")

# Visualize frequency spectra of original and filtered audio
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot filter masks
axes[0].plot(frequencies_audio, audio_lpf_mask, 'b-', label='LPF', linewidth=2)
axes[0].plot(frequencies_audio, audio_hpf_mask, 'r-', label='HPF', linewidth=2)
axes[0].axvline(x=audio_D0, color='g', linestyle='--', label=f'D0={audio_D0}Hz')
axes[0].set_xlim([0, sr/2])
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Filter Transfer Functions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot original spectrum
magnitude_original = np.abs(y_fft)
axes[1].semilogy(frequencies_audio, magnitude_original, 'b-', linewidth=0.5)
axes[1].axvline(x=audio_D0, color='r', linestyle='--', label=f'D0={audio_D0}Hz')
axes[1].set_xlim([0, sr/2])
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude (log)')
axes[1].set_title('Original Audio Spectrum')
axes[1].grid(True, alpha=0.3)

# Plot LPF filtered spectrum
magnitude_lpf = np.abs(np.fft.rfft(y_lpf))
axes[2].semilogy(frequencies_audio, magnitude_lpf, 'b-', linewidth=0.5)
axes[2].axvline(x=audio_D0, color='r', linestyle='--', label=f'D0={audio_D0}Hz')
axes[2].set_xlim([0, sr/2])
axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_ylabel('Magnitude (log)')
axes[2].set_title('LPF Filtered Audio Spectrum')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("""
NOTE: To hear the filtered audio:
- Use: librosa.output.write_wav('filtered.wav', y_lpf, sr)
- LPF removes high-frequency noise/hiss
- HPF removes low-frequency rumble
""")

print("\n" + "=" * 70)
print("AUDIO RESTORATION COMPLETE")
print("=" * 70)

In [ ]:
# Cell 12: Display Audio Waveform
# Normalize audio to [-1, 1] range and plot waveform

print("=" * 70)
print("AUDIO WAVEFORM - Time Domain Representation")
print("=" * 70)

# Normalize audio to [-1, 1] range
y_normalized = y / np.max(np.abs(y))

print(f"Normalized range: [{y_normalized.min():.4f}, {y_normalized.max():.4f}]")

# Plot waveform
plt.figure(figsize=(14, 4))
librosa.display.waveshow(y_normalized, sr=sr)
plt.title("Audio Waveform - Song with 2 Piccolo")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nWaveform shows amplitude variation over time.")
print("The piccolo melody creates distinct amplitude patterns.")

In [ ]:
# Cell 13: STFT and Spectrogram - Time-Frequency Analysis
# Compute Short-Time Fourier Transform for spectrogram

print("=" * 70)
print("SPECTROGRAM - Time-Frequency Analysis")
print("=" * 70)

# Compute Short-Time Fourier Transform (STFT)
D = librosa.stft(y)

print(f"STFT shape: {D.shape}")
print(f"STFT dtype: {D.dtype}")

# Convert to dB scale for better visualization
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

print(f"dB range: [{S_db.min():.2f}, {S_db.max():.2f}] dB")

# Plot spectrogram
plt.figure(figsize=(14, 5))
librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='hz', cmap='magma')
plt.colorbar(format='%+2.0f dB')
plt.title("Spectrogram - Song with 2 Piccolo")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.tight_layout()
plt.show()

print("\nSpectrogram shows frequency content over time.")
print("Brighter colors = higher energy at that frequency.")
print("The horizontal bands show the harmonic content of the piccolo.")

In [ ]:
# Cell 14: Spectral Analysis - Identify Dominant Frequencies
# Compute power spectral density and find dominant frequencies

print("=" * 70)
print("SPECTRAL ANALYSIS - Dominant Frequency Identification")
print("=" * 70)

# Compute mean spectrum across time
mean_spectrum = np.mean(np.abs(D), axis=1)
freqs = librosa.fft_frequencies(sr=sr)

# Plot power spectral density
plt.figure(figsize=(14, 4))
plt.semilogy(freqs, mean_spectrum)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Mean Magnitude')
plt.title('Average Power Spectral Density')
plt.grid(True, alpha=0.3)
plt.xlim([0, sr/2])
plt.tight_layout()
plt.show()

# Find peak frequencies (dominant harmonics)
from scipy.signal import find_peaks

peaks, properties = find_peaks(mean_spectrum, height=np.max(mean_spectrum)*0.1)
peak_frequencies = freqs[peaks]
peak_magnitudes = mean_spectrum[peaks]

# Sort by magnitude and show top 10
sorted_idx = np.argsort(peak_magnitudes)[::-1][:10]
print("\nTop 10 Dominant Frequencies:")
print("-" * 40)
for i, idx in enumerate(sorted_idx):
    print(f"{i+1:2d}. {peak_frequencies[idx]:7.1f} Hz  (magnitude: {peak_magnitudes[idx]:.4f})")

print(f"\nPiccolo frequency range: 500 Hz - 4 kHz")
print(f"Dominant peaks in piccolo range: {sum((peak_frequencies >= 500) & (peak_frequencies <= 4000))}")

In [ ]:
# Cell 15: Frequency Mixer - Creative Image Fusion
# Fuse two images: one provides structure (low freq), other provides details (high freq)

print("=" * 70)
print("FREQUENCY MIXER - Creative Image Fusion System")
print("=" * 70)
print("""
SYSTEM DESIGN:
==============
The Frequency Mixer combines frequency components from two different images:
- Image A provides LOW frequencies (overall structure/shape)
- Image B provides HIGH frequencies (fine details/texture)

This creates HYBRID images with mixed perceptual information.

MATHEMATICAL FORMULATION:
=========================
H_LPF(u,v) = 1  if sqrt(u² + v²) <= D0 (pass low frequencies)
           = 0  otherwise

H_HPF(u,v) = 0  if sqrt(u² + v²) <= D0 (block low frequencies)  
           = 1  otherwise

Mixed_Image(u,v) = F_A(u,v) × H_LPF(u,v) + F_B(u,v) × H_HPF(u,v)

Where F_A and F_B are the FFTs of the two source images.
""")

# Define cutoff frequency for optimal separation
D0 = min(cat_array.shape) // 8  # 12.5% of image dimension

print(f"Cutoff Frequency D0 = {D0} pixels")
print(f"  - Low freq region: radius <= {D0} pixels (structure)")
print(f"  - High freq region: radius > {D0} pixels (details)")

# Create frequency domain filter masks
M, N = cat_array.shape
u = np.arange(-M//2, M//2)
v = np.arange(-N//2, N//2)
V, U = np.meshgrid(v, u)
D = np.sqrt(U**2 + V**2)

lpf_mask = (D <= D0).astype(float)   # Low-pass: structure
hpf_mask = (D > D0).astype(float)    # High-pass: details

# Compute FFT of both images (centered)
cat_fft = fftshift(fft2(cat_array))
dog_fft = fftshift(fft2(dog_array))

# Create mixed images
# Mix 1: Cat structure + Dog details
mixed_fft_1 = (cat_fft * lpf_mask) + (dog_fft * hpf_mask)
mixed_1 = np.real(ifft2(fftshift(mixed_fft_1)))

# Mix 2: Dog structure + Cat details  
mixed_fft_2 = (dog_fft * lpf_mask) + (cat_fft * hpf_mask)
mixed_2 = np.real(ifft2(fftshift(mixed_fft_2)))

# Display the frequency mixer system - Creative Visualization
fig = plt.figure(figsize=(18, 12))

# Row 1: Source images and filter masks
ax1 = fig.add_subplot(3, 4, 1)
ax1.imshow(cat_array, cmap='gray')
ax1.set_title('Image A: Cat\n(provides STRUCTURE)', fontsize=10)
ax1.axis('off')

ax2 = fig.add_subplot(3, 4, 2)
ax2.imshow(dog_array, cmap='gray')
ax2.set_title('Image B: Dog\n(provides DETAILS)', fontsize=10)
ax2.axis('off')

ax3 = fig.add_subplot(3, 4, 3)
im3 = ax3.imshow(lpf_mask, cmap='Blues')
ax3.set_title(f'LPF Mask (D0={D0})\nStructure Filter', fontsize=10)
ax3.axis('off')
plt.colorbar(im3, ax=ax3, fraction=0.046)

ax4 = fig.add_subplot(3, 4, 4)
im4 = ax4.imshow(hpf_mask, cmap='Oranges')
ax4.set_title(f'HPF Mask (D0={D0})\nDetails Filter', fontsize=10)
ax4.axis('off')
plt.colorbar(im4, ax=ax4, fraction=0.046)

# Row 2: FFT magnitude spectra
cat_mag = np.log(1 + np.abs(cat_fft))
dog_mag = np.log(1 + np.abs(dog_fft))
mixed_mag_1 = np.log(1 + np.abs(mixed_fft_1))

ax5 = fig.add_subplot(3, 4, 5)
ax5.imshow(cat_mag, cmap='gray')
ax5.set_title('FFT: Cat\n(Low freq dominant)', fontsize=10)
ax5.axis('off')

ax6 = fig.add_subplot(3, 4, 6)
ax6.imshow(dog_mag, cmap='gray')
ax6.set_title('FFT: Dog\n(High freq dominant)', fontsize=10)
ax6.axis('off')

ax7 = fig.add_subplot(3, 4, 7)
ax7.imshow(lpf_mask * cat_mag, cmap='gray')
ax7.set_title(f'Cat × LPF\n(Low freq extracted)', fontsize=10)
ax7.axis('off')

ax8 = fig.add_subplot(3, 4, 8)
ax8.imshow(hpf_mask * dog_mag, cmap='gray')
ax8.set_title(f'Dog × HPF\n(High freq extracted)', fontsize=10)
ax8.axis('off')

# Row 3: Final mixed results
ax9 = fig.add_subplot(3, 4, 9)
ax9.imshow(cat_array, cmap='gray')
ax9.set_title('Original: Cat', fontsize=10)
ax9.axis('off')

ax10 = fig.add_subplot(3, 4, 10)
ax10.imshow(dog_array, cmap='gray')
ax10.set_title('Original: Dog', fontsize=10)
ax10.axis('off')

ax11 = fig.add_subplot(3, 4, 11)
ax11.imshow(np.clip(mixed_1, 0, 255), cmap='gray')
ax11.set_title('🎨 MIXED: Cat STRUCTURE\n+ Dog DETAILS', fontsize=10, color='blue')
ax11.axis('off')

ax12 = fig.add_subplot(3, 4, 12)
ax12.imshow(np.clip(mixed_2, 0, 255), cmap='gray')
ax12.set_title('🎨 MIXED: Dog STRUCTURE\n+ Cat DETAILS', fontsize=10, color='blue')
ax12.axis('off')

plt.tight_layout()
plt.show()

# Plot 2D Transfer Functions as 3D surfaces
fig = plt.figure(figsize=(18, 5))

# LPF Transfer Function (3D surface)
ax1 = fig.add_subplot(131, projection='3d')
Z_lpf = lpf_mask
ax1.plot_surface(V, U, Z_lpf, cmap='Blues', alpha=0.9)
ax1.set_title('LPF Transfer Function\nH_LPF(u,v) = 1 inside D0', fontsize=11)
ax1.set_xlabel('u (frequency)')
ax1.set_ylabel('v (frequency)')
ax1.set_zlabel('H(u,v)')
ax1.view_init(elev=30, azim=45)

# HPF Transfer Function (3D surface)
ax2 = fig.add_subplot(132, projection='3d')
Z_hpf = hpf_mask
ax2.plot_surface(V, U, Z_hpf, cmap='Oranges', alpha=0.9)
ax2.set_title('HPF Transfer Function\nH_HPF(u,v) = 1 outside D0', fontsize=11)
ax2.set_xlabel('u (frequency)')
ax2.set_ylabel('v (frequency)')
ax2.set_zlabel('H(u,v)')
ax2.view_init(elev=30, azim=45)

# Combined Mixer Transfer Function
ax3 = fig.add_subplot(133, projection='3d')
Z_mixer = np.where(D <= D0, 1, 2).astype(float)
ax3.plot_surface(V, U, Z_mixer, cmap='coolwarm', alpha=0.9)
ax3.set_title('Mixer Transfer Function\n1=Structure, 2=Details', fontsize=11)
ax3.set_xlabel('u (frequency)')
ax3.set_ylabel('v (frequency)')
ax3.set_zlabel('Source Image')
ax3.view_init(elev=30, azim=45)

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("FREQUENCY MIXER - Analysis & Observations")
print("=" * 70)
print(f"""
RESULTS:
========
Mix 1 (Cat structure + Dog details):
  - Contains the overall SHAPE/FORM of the cat
  - Surface TEXTURE/DETAILS from the dog

Mix 2 (Dog structure + Cat details):
  - Contains the overall SHAPE/FORM of the dog
  - Surface TEXTURE/DETAILS from the cat

TRANSFER FUNCTIONS:
===================
|  Region  |    Radius    | Passes | Contains |
|----------|-------------|--------|----------|
| Low Freq | r <= {D0:4d} px | LPF   | Structure |
| High Freq| r >  {D0:4d} px | HPF   | Details   |

APPLICATIONS:
=============
• Image blending and artistic effects
• Medical imaging (CT + MRI fusion)
• Multi-focus photography combination
• Texture transfer between images
""")

print("=" * 70)
print("FREQUENCY MIXER COMPLETE")
print("=" * 70)

In [ ]:
# Cell 16: Summary and Conclusions
print("=" * 75)
print("       EE200 SIGNAL PROCESSING PROJECT - COMPLETE SUMMARY")
print("=" * 75)

print("\n" + "-" * 75)
print("PART A: IMAGE PROCESSING")
print("-" * 75)
print(f"  1. Basic Operations: Load, resize (200x200), crop, rotate (45°)")
print(f"  2. 2D DFT: FFT computed, magnitude & phase spectra displayed")
print(f"  3. Frequency Filtering: LPF (6.25%, 12.5%, 25%) and HPF")
print(f"  4. Rotation Analysis: 90° anti-clockwise FFT comparison")
print(f"  5. Frequency Mixer: Creative image fusion with transfer functions")

print("\n" + "-" * 75)
print("PART B: AUDIO PROCESSING")  
print("-" * 75)
print(f"  1. Audio Loading: sr={sr} Hz, duration={duration:.2f}s")
print(f"  2. Audio Restoration: FFT-based LPF/HPF filtering")
print(f"  3. Waveform: Normalized amplitude vs time plot")
print(f"  4. Spectrogram: STFT with dB scale visualization")
print(f"  5. Spectral Analysis: Top 10 dominant frequencies identified")

print("\n" + "=" * 75)
print("KEY CONCEPTS & FORMULAS")
print("=" * 75)
print("""
1. 2D DISCRETE FOURIER TRANSFORM (DFT):
   F(u,v) = Σₓ Σᵧ f(x,y) × e^(-j2π(ux/M + vy/N))
   
2. IDEAL LOW-PASS FILTER (LPF):
   H_LPF(u,v) = 1 if sqrt(u²+v²) ≤ D₀, else 0
   
3. IDEAL HIGH-PASS FILTER (HPF):
   H_HPF(u,v) = 0 if sqrt(u²+v²) ≤ D₀, else 1
   
4. FREQUENCY MIXER (Image Fusion):
   F_mixed(u,v) = F_A(u,v)×H_LPF(u,v) + F_B(u,v)×H_HPF(u,v)
   
5. STFT FOR AUDIO:
   D(k,l) = Σₙ x(n) × w(n-lR) × e^(-j2πkn/N)
""")

print("\n" + "=" * 75)
print("           PROJECT COMPLETED SUCCESSFULLY!")
print("=" * 75)